# B2-023-generative-models-diffusion — Practice p03 — Solution

**Type:** mc · **Difficulty:** intro · **Concepts:** denoising-diffusion-probabilistic-models

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

**Alphas.** $\alpha_t=1-\beta_t$: $\alpha_1=0.8$, $\alpha_2=0.75$, $\alpha_3=0.5$.

**Cumulative products.** $\bar\alpha_1=0.8$, $\bar\alpha_2=0.8\cdot0.75=0.6$, $\bar\alpha_3=0.6\cdot0.5=0.3$.

**Answer: C** ($\bar\alpha_3=0.3$).

**Conditional distribution at $x_0=2$.** $q(x_3\mid x_0)=N(\sqrt{\bar\alpha_3}\,x_0,\ 1-\bar\alpha_3)$, so
mean $=\sqrt{0.3}\cdot2=1.095445$ and standard deviation $=\sqrt{1-0.3}=\sqrt{0.7}=0.836660$ (the variance is $0.7$; the standard deviation is its square root).

**What each wrong option computes.**

* A $=0.025=\beta_1\beta_2\beta_3$: the product of the **betas** instead of the alphas.
* B $=0.05=1-(\beta_1+\beta_2+\beta_3)$: subtracting the betas instead of multiplying the alphas (the linearization $\prod(1-\beta_t)\approx1-\sum\beta_t$); it also equals $\beta_1\beta_2$.
* D $=0.5=\alpha_3$: the last single-step $\alpha$, the $\alpha$/$\bar\alpha$ confusion.
* E $=0.7=1-\bar\alpha_3$: the **noise** variance at $t=3$, not the signal factor.

In [ ]:
import math

beta = [0.2, 0.25, 0.5]
alpha = [1 - b for b in beta]
alpha_bar = []
running = 1.0
for a in alpha:
    running *= a
    alpha_bar.append(running)

x0 = 2.0
cond_mean = math.sqrt(alpha_bar[2]) * x0
cond_std = math.sqrt(1 - alpha_bar[2])
options = {"A": 0.025, "B": 0.05, "C": 0.3, "D": 0.5, "E": 0.7}
ANSWER = [k for k, v in options.items() if math.isclose(v, alpha_bar[2], abs_tol=1e-12, rel_tol=1e-09)][0]
print("alpha", alpha, "alpha_bar", alpha_bar)
print(f"mean {cond_mean:.6f}  std {cond_std:.6f}  answer {ANSWER}")

### Answer check

In [ ]:
assert all(math.isclose(a, b, abs_tol=1e-12, rel_tol=1e-09) for a, b in zip(alpha, [0.8, 0.75, 0.5]))
assert math.isclose(alpha_bar[0], 0.8, abs_tol=1e-12, rel_tol=1e-09) and math.isclose(alpha_bar[1], 0.6, abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(alpha_bar[2], 0.3, abs_tol=1e-12, rel_tol=1e-09)
assert round(cond_mean, 6) == 1.095445 and round(cond_std, 6) == 0.836660
assert math.isclose(beta[0] * beta[1] * beta[2], options["A"], abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(1 - sum(beta), options["B"], abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(alpha[2], options["D"], abs_tol=1e-12, rel_tol=1e-09)
assert math.isclose(1 - alpha_bar[2], options["E"], abs_tol=1e-12, rel_tol=1e-09)
assert ANSWER == "C"